# Video Sentiment Pipeline - Colab Notebook

This notebook runs the project end-to-end with checkpoints, resume support, plots, and evaluation.

It supports:
- Current one-class setup (`normal`) with synthetic violent labels for debugging.
- Placeholder binary setup using one violent folder: `Violent_Videos_Placeholder/`.
- Taxonomy extraction from `UCFCrime_Filtered_WithFilename.json`.
- Auto-resume training in Google Drive using `checkpoint_last.pt`.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB =', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')
else:
    print('Running outside Colab; Drive mount skipped.')


In [ ]:
# Repo resolution: use current directory if already inside the repo.
CWD = Path.cwd()
if (CWD / 'train_video_classifier.py').exists():
    REPO_DIR = CWD
else:
    REPO_DIR = Path('/content/transverse_project_idemia')

REPO_GIT_URL = ''  # Optional: set if repo is not already available.

if not REPO_DIR.exists():
    if REPO_GIT_URL:
        import subprocess
        subprocess.run(['git', 'clone', REPO_GIT_URL, str(REPO_DIR)], check=True)
    else:
        raise FileNotFoundError(
            f'Repo directory not found: {REPO_DIR}. Set REPO_GIT_URL or upload repo first.'
        )

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())


In [ ]:
# Install runtime dependencies.
import subprocess

subprocess.run(['bash', 'scripts/colab_setup.sh'], check=True)

INSTALL_MMACTION2 = False
if INSTALL_MMACTION2:
    subprocess.run('pip install -U openmim', shell=True, check=True)
    subprocess.run('mim install mmengine', shell=True, check=True)
    subprocess.run('mim install "mmcv>=2.0.0"', shell=True, check=True)
    subprocess.run('mim install mmaction2', shell=True, check=True)
    print('MMAction2 installed.')
else:
    print('MMAction2 install skipped (set INSTALL_MMACTION2=True if needed).')


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## Data Paths

Set these paths if your data is stored elsewhere (for example in Google Drive).


In [ ]:
from pathlib import Path

NORMAL_DIR = Path('Normal_Videos_for_Event_Recognition/Normal_Videos_for_Event_Recognition').resolve()
VIOLENT_DIR = Path('Violent_Videos_Placeholder').resolve()
FILTERED_JSON = Path('UCFCrime_Filtered_WithFilename.json').resolve()
MANIFEST_DIR = Path('manifests').resolve()
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print('NORMAL_DIR:', NORMAL_DIR, 'exists=', NORMAL_DIR.exists())
print('VIOLENT_DIR:', VIOLENT_DIR, 'exists=', VIOLENT_DIR.exists())
print('FILTERED_JSON:', FILTERED_JSON, 'exists=', FILTERED_JSON.exists())

normal_mp4 = list(NORMAL_DIR.rglob('*.mp4')) if NORMAL_DIR.exists() else []
violent_mp4 = list(VIOLENT_DIR.rglob('*.mp4')) if VIOLENT_DIR.exists() else []
print('normal mp4 count =', len(normal_mp4))
print('violent mp4 count =', len(violent_mp4))


In [ ]:
# 1) Extract class taxonomy from UCFCrime filtered JSON.
import subprocess

subprocess.run([
    'python', 'scripts/extract_ucfcrime_taxonomy.py',
    '--input-json', str(FILTERED_JSON),
    '--output-json', 'manifests/ucfcrime_taxonomy.json'
], check=True)


In [ ]:
# 2) Build bootstrap manifests for one-class data (synthetic violent labels for testing).
import subprocess

subprocess.run([
    'python', 'scripts/build_bootstrap_manifests.py',
    '--normal-dir', str(NORMAL_DIR),
    '--taxonomy-json', 'manifests/ucfcrime_taxonomy.json',
    '--output-dir', 'manifests',
    '--max-normal', '120',
    '--synthetic-train-per-class', '2',
    '--synthetic-val-per-class', '1',
    '--synthetic-test-per-class', '1',
    '--infer-duration'
], check=True)


In [ ]:
# 3) Build production-like binary manifest if violent folder has data.
# This assumes one violent folder as requested.
import subprocess

violent_mp4 = list(VIOLENT_DIR.rglob('*.mp4')) if VIOLENT_DIR.exists() else []
if len(violent_mp4) == 0:
    print('No violent videos found in placeholder folder. Skipping binary production manifest build.')
    print('Add MP4 files to:', VIOLENT_DIR)
else:
    subprocess.run([
        'python', 'scripts/build_video_manifest.py',
        '--class-dir', f'normal={NORMAL_DIR}',
        '--class-dir', f'violent={VIOLENT_DIR}',
        '--output', 'manifests/production_binary_manifest.csv',
        '--infer-duration'
    ], check=True)
    print('Wrote manifests/production_binary_manifest.csv')


In [ ]:
# 4) Plot class/split distributions from manifest files.
import csv
from collections import Counter
import matplotlib.pyplot as plt

def load_manifest_counts(csv_path):
    rows = list(csv.DictReader(open(csv_path, 'r', encoding='utf-8')))
    by_split = Counter(r['split'] for r in rows)
    by_label = Counter(r['label'] for r in rows)
    return rows, by_split, by_label

def plot_counter(counter, title):
    labels = list(counter.keys())
    values = [counter[k] for k in labels]
    plt.figure(figsize=(10, 4))
    plt.bar(labels, values)
    plt.title(title)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

for name in [
    'manifests/bootstrap_binary_manifest.csv',
    'manifests/bootstrap_multiclass_manifest.csv',
    'manifests/bootstrap_multilabel_manifest.csv',
    'manifests/production_binary_manifest.csv',
]:
    p = Path(name)
    if not p.exists():
        print('skip (missing):', p)
        continue
    rows, split_counts, label_counts = load_manifest_counts(p)
    print('\n', p, 'rows=', len(rows))
    print('split counts:', dict(split_counts))
    print('label count (top 20):', dict(label_counts.most_common(20)))
    plot_counter(split_counts, f'{p.name} - split distribution')
    plot_counter(label_counts, f'{p.name} - label distribution')


## Training

Start with smoke runs, then use Colab auto-resume for longer training.


In [ ]:
# 5) Quick binary smoke run from bootstrap manifest.
import subprocess

subprocess.run([
    'python', 'train_video_classifier.py',
    '--config', 'configs/smoke_bootstrap_binary.yaml',
    '--manifest', 'manifests/bootstrap_binary_manifest.csv',
    '--output-dir', 'runs/notebook_smoke_bootstrap_binary'
], check=True)


In [ ]:
# 6) Long training with auto-resume in persistent storage.
# Rerun this cell after disconnects; it resumes from checkpoint_last.pt automatically.
import subprocess
from pathlib import Path

RUN_FULL_TRAINING = False  # Set True when you are ready.
TASK = 'multiclass'  # 'binary' | 'multiclass' | 'multilabel'

if TASK == 'binary':
    base_config = 'configs/template_binary.yaml'
    manifest = 'manifests/production_binary_manifest.csv'
elif TASK == 'multiclass':
    base_config = 'configs/template_ucfcrime_multiclass_from_taxonomy.yaml'
    manifest = 'manifests/bootstrap_multiclass_manifest.csv'
else:
    base_config = 'configs/template_ucfcrime_multilabel_from_taxonomy.yaml'
    manifest = 'manifests/bootstrap_multilabel_manifest.csv'

default_drive_dir = Path('/content/drive/MyDrive/violence_runs')
if default_drive_dir.exists():
    RUN_DIR = default_drive_dir / f'{TASK}_run_01'
else:
    RUN_DIR = Path('runs') / f'{TASK}_run_01'

cmd = [
    'python', 'scripts/colab_autoresume.py',
    '--config', base_config,
    '--run-dir', str(RUN_DIR),
    '--manifest', manifest,
    '--class-names-file', 'manifests/ucfcrime_taxonomy.json'
]

print('Command:', ' '.join(cmd))
if RUN_FULL_TRAINING:
    subprocess.run(cmd, check=True)
else:
    print('RUN_FULL_TRAINING=False, dry-run only.')
    subprocess.run(cmd + ['--dry-run'], check=True)


In [ ]:
# 7) Evaluate best checkpoint and plot per-class F1.
import json
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt

EVAL_CONFIG = 'configs/template_ucfcrime_multiclass_from_taxonomy.yaml'
EVAL_MANIFEST = 'manifests/bootstrap_multiclass_manifest.csv'
EVAL_RUN_DIR = Path('runs/notebook_eval_example')  # set to your real run dir
BEST_CKPT = EVAL_RUN_DIR / 'checkpoint_best.pt'

if not BEST_CKPT.exists():
    print('Checkpoint not found:', BEST_CKPT)
    print('Set EVAL_RUN_DIR to the folder that contains checkpoint_best.pt')
else:
    subprocess.run([
        'python', 'test_video_classifier.py',
        '--config', EVAL_CONFIG,
        '--manifest', EVAL_MANIFEST,
        '--checkpoint', str(BEST_CKPT),
        '--split', 'test'
    ], check=True)

    metrics_path = EVAL_RUN_DIR / 'metrics_test_from_checkpoint_best.json'
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        classes = [row['class_name'] for row in metrics.get('per_class', [])]
        f1 = [row['f1'] for row in metrics.get('per_class', [])]
        plt.figure(figsize=(12, 4))
        plt.bar(classes, f1)
        plt.title('Per-class F1 (test)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print('Metrics file not found:', metrics_path)


In [ ]:
# 8) Compare model runs (macro-F1 ranking).
import subprocess

RUN_DIRS = [
    # 'runs/ucfcrime_multiclass_r3d18',
    # 'runs/mmaction2_multiclass',
]

if len(RUN_DIRS) < 2:
    print('Add at least two run directories in RUN_DIRS to compare.')
else:
    cmd = ['python', 'scripts/compare_runs.py']
    for rd in RUN_DIRS:
        cmd.extend(['--run-dir', rd])
    cmd.extend(['--output-csv', 'runs/notebook_model_comparison.csv'])
    subprocess.run(cmd, check=True)


## Notes for Team Handover

- Use bootstrap manifests only for debugging (synthetic labels).
- For final reporting, train only on real labels/data.
- Keep `checkpoint_last.pt` on persistent storage (Drive or cloud bucket).
- Re-run taxonomy extraction when updated JSON annotation files arrive.
